# English-Luganda Translator - ML Pipeline on Colab

**GPU-Accelerated Machine Learning Workflow**

This notebook trains an English-Luganda translator using:
- Week 2: ML Workflow (load → preprocess → train → evaluate)
- Week 3: Regularization (dropout, L2)
- Week 6: Evaluation Metrics (BLEU score)
- Week 9: Transformers (sequence-to-sequence)

**Datasets**: 5 parallel corpora (3100+ pairs)
**GPU**: Free Tesla T4 on Colab (5-10x faster than CPU)
**Time**: ~15-20 minutes total

## STEP 1: Setup Environment

Install required packages

In [ ]:
print("="*80)
print("ENGLISH-LUGANDA TRANSLATOR - COLAB ML PIPELINE")
print("="*80)
print("\n[STEP 1: Installing packages]\n")

import subprocess
import sys

packages = [
    "torch",
    "transformers",
    "datasets",
    "pandas",
    "scikit-learn",
    "sacrebleu",
]

print("📦 Installing packages...")
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    print(f"  ✓ {package}")

print("\n✅ All packages installed")

## STEP 2: Mount Google Drive & Setup Project

Connect to your Google Drive where the project is uploaded

In [ ]:
print("\n[STEP 2: Setting up project]\n")

from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
print("📁 Mounting Google Drive...")
drive.mount('/content/drive')
print("✅ Drive mounted\n")

# Set working directory
# NOTE: Adjust this path based on where you uploaded the project
COLAB_PROJECT_PATH = "/content/drive/My Drive/English-Luganda-Translator/ENGLISH-LUGANDA-TRANSLATOR"

if not os.path.exists(COLAB_PROJECT_PATH):
    print(f"⚠️  Path not found: {COLAB_PROJECT_PATH}")
    print("\nIf your folder is at a different location:")
    print("  1. Find the folder path in Google Drive")
    print("  2. Update COLAB_PROJECT_PATH above")
    print("  3. Re-run this cell")
else:
    os.chdir(COLAB_PROJECT_PATH)
    print(f"✅ Working directory: {os.getcwd()}")

## STEP 3: Load All Datasets

Load and combine 5 English-Luganda parallel datasets

In [ ]:
print("\n[STEP 3: Loading datasets]\n")
print("="*80)

import sys
sys.path.insert(0, "src")

from load_data import load_all_datasets, get_dataset_statistics
from utils import print_section

# Load all datasets
combined_df = load_all_datasets()
stats = get_dataset_statistics(combined_df)

print(f"\n✅ Dataset Statistics:")
print(f"   Total samples: {stats['total_samples']:,}")
print(f"   English text - Avg length: {stats['avg_english_length']:.1f}")
print(f"   Luganda text - Avg length: {stats['avg_luganda_length']:.1f}")

## STEP 4: Preprocess & Create Splits

Clean text and create train (80%) / val (10%) / test (10%) splits

In [ ]:
print("\n[STEP 4: Preprocessing & creating splits]\n")
print("="*80)

from preprocess import preprocess_and_split, save_splits

# Preprocess and split
train_df, val_df, test_df = preprocess_and_split(combined_df)
save_splits(train_df, val_df, test_df)

print(f"\n✅ Data splits created:")
print(f"   Train: {len(train_df):,}")
print(f"   Val: {len(val_df):,}")
print(f"   Test: {len(test_df):,}")

## STEP 5: Train Model on GPU

⚠️ **This will take 8-12 minutes on GPU**

Fine-tune transformer model with your data

In [ ]:
print("\n[STEP 5: Training model]\n")
print("⚠️  This will take 8-12 minutes on GPU\n")

import torch

print(f"🤖 GPU Status:")
print(f"   Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   Device: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\n" + "="*80)
print("STARTING TRAINING")
print("="*80 + "\n")

from train import main as train_main

try:
    model, tokenizer = train_main()
    print(f"\n✅ Training complete!")
except Exception as e:
    print(f"❌ Training error: {e}")
    import traceback
    traceback.print_exc()

## STEP 6: Evaluate Model

Calculate BLEU score on test set

In [ ]:
print("\n[STEP 6: Evaluating model]\n")
print("="*80)

from evaluate import main as eval_main

try:
    eval_main()
    print(f"\n✅ Evaluation complete!")
except Exception as e:
    print(f"❌ Evaluation error: {e}")
    import traceback
    traceback.print_exc()

## STEP 7: Display Results

View BLEU score and metrics

In [ ]:
print("\n[STEP 7: Results Summary]\n")

import json
from pathlib import Path

eval_file = Path("outputs/evaluation_results.json")
if eval_file.exists():
    with open(eval_file) as f:
        results = json.load(f)
    
    print("\n" + "="*80)
    print("📊 FINAL RESULTS")
    print("="*80)
    print(f"\n✅ BLEU Score: {results['bleu_score']:.2f}")
    print(f"   Test samples: {results['num_test_samples']}")
    print(f"   Avg prediction length: {results['avg_prediction_length']:.1f} tokens")
    print(f"   Avg reference length: {results['avg_reference_length']:.1f} tokens")
else:
    print("⚠️  Results file not found")

## STEP 8: Download Results & Model

Download trained model and evaluation results

In [ ]:
print("\n[STEP 8: Downloading files]\n")

from google.colab import files
import shutil

print("📥 Preparing files for download...\n")

# Create zip of trained model
print("   Zipping trained model...")
shutil.make_archive("trained_model", "zip", "models")

# Create zip of results
print("   Zipping evaluation results...")
shutil.make_archive("evaluation_outputs", "zip", "outputs")

print("\n📥 Download files:")
print("   1. trained_model.zip - Use for inference")
print("   2. evaluation_outputs.zip - BLEU scores and predictions")
print("\nClicking download button...")

files.download("trained_model.zip")
files.download("evaluation_outputs.zip")

print("\n✅ Files ready for download!")

## STEP 9 (Optional): Test Inference

Test the trained model with sample sentences

In [ ]:
print("\n[STEP 9: Testing inference (Optional)]\n")

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

print("🧪 Testing model inference...\n")

try:
    model_path = "models/trained_model"
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
    
    # Test translations
    test_sentences = [
        "Hello, how are you?",
        "Thank you very much.",
        "What is your name?",
    ]
    
    print("📝 Sample Translations:\n")
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    
    for sentence in test_sentences:
        inputs = tokenizer(sentence, return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        output_ids = model.generate(
            inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=128,
        )
        
        translation = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        print(f"   English:  {sentence}")
        print(f"   Luganda:  {translation}\n")
        
except Exception as e:
    print(f"⚠️  Inference test failed: {e}")

## ✅ Pipeline Complete!

### Summary
- ✓ Step 1: Loaded all 5 datasets (3100+ pairs)
- ✓ Step 2: Created train/val/test splits (80/10/10)
- ✓ Step 3: Trained model on GPU (5-15 minutes)
- ✓ Step 4: Evaluated on test set (BLEU score)
- ✓ Step 5: Downloaded trained model & results

### Results
- **BLEU Score**: See above
- **Training Time**: ~10 minutes on GPU
- **Model**: `trained_model.zip` (ready to download)
- **Predictions**: `evaluation_outputs.zip` (ready to download)

### Next Steps
1. Download the trained model
2. Extract and use for inference locally
3. Fine-tune further if needed
4. Deploy to production

---

**Course Alignment**: Week 2 (ML Workflow) → Week 3 (Regularization) → Week 6 (Metrics) → Week 9 (Transformers)

**Status**: ✅ Ready for training!